In [11]:
import re
import subprocess
from pathlib import Path
import pymupdf
from uuid import uuid4
import pymupdf4llm
from shared import embedder


In [2]:
import re
import subprocess
from pathlib import Path


from shared import embedder

DOCS_DIR = Path("../documents")


In [3]:
def ensure_search_indexes(collection) -> None:
    existing = {index["name"] for index in collection.list_search_indexes()}
    created = []
    collection.create_search_index(
        {
            "name": "knowledge_vector_index",
            "type": "vectorSearch",
            "definition": {
                "fields": [
                    {
                        "type": "vector",
                        "path": "embedding",
                        "numDimensions": EMBEDDING_DIMENSION,
                        "similarity": "cosine",
                    }
                ]
            },
        }
    )
    collection.create_search_index(
        {
            "name": "knowledge_text_index",
            "definition": {
                "mappings": {
                    "dynamic": False,
                    "fields": {
                        "text": {"type": "string", "analyzer": "lucene.russian"},
                        "section_title": {"type": "string", "analyzer": "lucene.russian"},
                    },
                }
            },
        }
    )
    print(f"Waiting for search indexes to build: {created}")
    deadline = tm.monotonic() + 180
    while tm.monotonic() < deadline:
        statuses = {index["name"]: index for index in collection.list_search_indexes()}
        if all(statuses.get(name, {}).get("queryable") for name in created):
            print("Search indexes are queryable.")
            return
        tm.sleep(2)
    raise TimeoutError(f"Search indexes did not become queryable in time: {created}")

In [30]:
import hashlib
import json
import os
import re
import time as tm
from concurrent.futures import ThreadPoolExecutor, as_completed, ProcessPoolExecutor
from pathlib import Path

import pymongo
import pymongo.errors
import pymupdf4llm
from langchain_text_splitters import RecursiveCharacterTextSplitter
from shared import embedder

# Must match backend/src/storages/mongo/knowledge.py's KnowledgeChunk.Settings —
# duplicated, not imported, since backend/ingest deliberately doesn't depend on
# the backend package (see [[backend-ingest-separate-project]]).
COLLECTION_NAME = "knowledge_chunks"
VECTOR_INDEX_NAME = "knowledge_vector_index"
TEXT_INDEX_NAME = "knowledge_text_index"
EMBEDDING_DIMENSION = 1024  # mxbai-embed-large
MONGO_URI =  "mongodb://localhost/db?directConnection=true"


client = pymongo.MongoClient(MONGO_URI)
collection = client.get_database()[COLLECTION_NAME]

In [5]:
import re
from langchain_text_splitters import RecursiveCharacterTextSplitter
# mxbai-embed-large's Ollama context window is 512 tokens; Cyrillic text runsq
# well under 1 char/token, so keep chunks well short of that to avoid MV_HTTP 500s.
CHUNK_SIZE = 400
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=40)

# document_path = Path("./documents/Инструкция_по_работе_с_машиночитаемыми_доверенностями.pdf")


# page_count = pymupdf.open(document_path).page_count
# doc = pymupdf4llm.to_markdown(document_path, pages=list(range(2, page_count)))





In [41]:
document_path

PosixPath('documents/Инструкция_по_работе_с_машиночитаемыми_доверенностями.pdf')

In [78]:
from pprint import pprint
from uuid import uuid4
splitted = splitter.split_text(doc)


items = [
                    {
                    "id": str(uuid4()),
                    "text": item,
                    "path": str(document_path),
                    "label": "manual",
                    "term": None,
                }
                for item in splitted
]

In [6]:
def embed_and_insert_batch(collection, batch: list[dict]) -> tuple[int, int]:
    """Embed one batch and save it immediately, so progress lands in Mongo as
    each batch finishes rather than only after every batch has embedded."""
    embeddings = embedder.embed_documents([item["text"] for item in batch])
    for item, embedding in zip(batch, embeddings, strict=True):
        item["embedding"] = [float(x) for x in embedding]


    result = collection.insert_many(batch, ordered=False)
    return len(result.inserted_ids), 0

KeyError: '_id'

In [103]:
# embed_and_insert_batch(collection, items)
result = collection.insert_many(items)

In [54]:

def embed_and_insert_in_pool(data):
    BATCH_SIZE = 32
    MAX_WORKERS = 6

    batches = [data[i : i + BATCH_SIZE] for i in range(0, len(data), BATCH_SIZE)]
    inserted_total = 0
    duplicate_total = 0
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        futures = [pool.submit(embed_and_insert_batch, collection, batch) for batch in batches]
        for done, future in enumerate(as_completed(futures), start=1):
            inserted, duplicates = future.result()
            inserted_total += inserted
            duplicate_total += duplicates
            print(f"Batch {done}/{len(batches)} done ({inserted} inserted, {duplicates} duplicate text skipped)")


from pprint import pprint
import json


def extract_table_glossary(data, glossary_source):
    extract = []
    for page in data.get("pages", []):
        for block in page.get("boxes", []):
            if block["boxclass"] == "table":
                extract.extend(block["table"]["extract"][1:])  # skip header row


    term_items = [
    {
        "id": str(uuid4()),
        "text": f"{term.strip()} — {definition.strip()}",
        "label": "glossary",
        "path": str(glossary_source),
        "term": term.strip(),
    }
    for term, definition in extract
    if term.strip() and definition.strip()
    ]
    return term_items



In [19]:
query = 'определении ИНН'
query_vector = embedder.embed_query(query)

vector_pipeline = [
    {
        "$vectorSearch": {
            "index": VECTOR_INDEX_NAME,
            "path": "embedding",
            "queryVector": query_vector,
            "numCandidates": 20,
            "limit": 20,
        }
    },
    {"$project": {"embedding": 0}},
]
text_pipeline = [
    {"$search": {"index": TEXT_INDEX_NAME, "text": {"query": query, "path": ["text", "section_title"]}}},
    {"$limit": 20},
    {"$project": {"embedding": 0}},
]
vector_hist = collection.aggregate(vector_pipeline).to_list(length=20)
text_hits = collection.aggregate(text_pipeline).to_list(length=20)


[{'_id': ObjectId('6aa5b8e9aba79fd8d4b4f27d'),
  'id': 'd40faa84-dfbd-456b-bca0-8dc270ee7722',
  'text': '- Начальная цена котировочной сессии; \n\n- Дата начала котировочной сессии; \n\n- Дата окончания котировочной сессии; \n\n- Наименование заказчика; \n\n- ИНН заказчика; \n\n- Законы, в соответствии с которыми осуществляется/осуществлялась закупка; \n\n- Основание заключения контракта; \n\n- Способ размещения закупки (заказа)/определение поставщика.',
  'path': 'documents/Инструкция_по_работе_с_Порталом_для_заказчика.pdf',
  'label': 'manual',
  'term': None},
 {'_id': ObjectId('6aa5b889aba79fd8d4b4f1d0'),
  'id': 'f1576800-0465-4528-b018-803f865e0e01',
  'text': '− Выбрать параметры получения для различных типов уведомлений (уведомления на Портале поставщиков либо email); \n\n− Выполнить подписку на получение уведомлений определенного типа (установить флажок); \n\n− Отменить подписку на получение уведомлений определенного типа.',
  'path': 'documents/Инструкция_по_работе_с_Портало

In [6]:
hints = vector_hist + text_hits
len(hints)

40

In [7]:
documents = [
    hit['text'] for hit in hints
]

In [10]:
documents[32]

'При нажатии на кнопку «Операции с МЧД» откроется модальное окно «Машиночитаемые доверенности пользователя» (Рисунок ), в котором администратору необходимо загрузить доверенность, добавив xml-файл в поле «Загрузить МЧД из файла» (Рисунок  (1)) и нажать на кнопку «Добавить» (Рисунок  (2)). \n\n\n\n**Рисунок 6 – Модальное окно «Машиночитаемые доверенности пользователя»**'

In [8]:
from sentence_transformers import CrossEncoder

reranker_model = CrossEncoder('DiTy/cross-encoder-russian-msmarco', max_length=512)


rank_result = reranker_model.rank(query, documents)
print(rank_result)
# `[{'corpus_id': 0, 'score': 0.88126713},
#  {'corpus_id': 2, 'score': 0.001042091},
#  {'corpus_id': 3, 'score': 0.0010417715},
#  {'corpus_id': 1, 'score': 0.0010344835},
#  {'corpus_id': 4, 'score': 0.0010244923}]`


'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/DiTy/cross-encoder-russian-msmarco/resolve/main/./modules.json
Retrying in 1s [Retry 1/5].
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/DiTy/cross-encoder-russian-msmarco/resolve/main/./modules.json
Retrying in 2s [Retry 2/5].
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/DiTy/cross-encoder-russian-msmarco/resolve/main/./modules.json
Retrying in 4s [Retry 3/5].
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/DiTy/cross-encoder-russian-msmarco/resolve/main/./modules.json
Retrying in 8s [Retry 4/5].
'[Errno -3] Temporary failure in name resolution' thrown while requesting HEAD https://huggingface.co/DiTy/cross-encoder-russian-msmarco/resolve/main/./modules.json
Retrying in 8s [Retry 5/5].
'[Errno -3] Temporary failure in na

[{'corpus_id': 32, 'score': 0.5695022344589233}, {'corpus_id': 5, 'score': 0.5100562572479248}, {'corpus_id': 22, 'score': 0.5100562572479248}, {'corpus_id': 12, 'score': 0.4832008183002472}, {'corpus_id': 29, 'score': 0.4832008183002472}, {'corpus_id': 0, 'score': 0.4391607642173767}, {'corpus_id': 23, 'score': 0.4391607642173767}, {'corpus_id': 13, 'score': 0.3190537691116333}, {'corpus_id': 24, 'score': 0.3190537691116333}, {'corpus_id': 10, 'score': 0.2179134488105774}, {'corpus_id': 26, 'score': 0.2179134488105774}, {'corpus_id': 4, 'score': 0.18631470203399658}, {'corpus_id': 21, 'score': 0.18631470203399658}, {'corpus_id': 2, 'score': 0.17089588940143585}, {'corpus_id': 20, 'score': 0.17089588940143585}, {'corpus_id': 27, 'score': 0.12966132164001465}, {'corpus_id': 17, 'score': 0.12813059985637665}, {'corpus_id': 25, 'score': 0.12813059985637665}, {'corpus_id': 3, 'score': 0.05208312347531319}, {'corpus_id': 28, 'score': 0.05208312347531319}, {'corpus_id': 34, 'score': 0.051384

In [ ]:
document_path = Path("./documents/Инструкция_по_работе_с_Порталом_для_заказчика.pdf")


page_count = pymupdf.open(document_path).page_count
json_string = pymupdf4llm.to_json(document_path, pages=[3])
term_items = extract_table_glossary(json.loads(json_string), document_path)
term_items
embed_and_insert_batch(collection, term_items)

(26, 0)

In [51]:
doc = pymupdf4llm.to_markdown(document_path, pages=list(range(4, page_count - 1)))
splitted = splitter.split_text(doc)


items = [
                    {
                    "id": str(uuid4()),
                    "text": item,
                    "path": str(document_path),
                    "label": "manual",
                    "term": None,
                }
                for item in splitted
]

In [52]:
embed_and_insert_in_pool(items)

Batch 1/24 done (32 inserted, 0 duplicate text skipped)
Batch 2/24 done (32 inserted, 0 duplicate text skipped)
Batch 3/24 done (32 inserted, 0 duplicate text skipped)
Batch 4/24 done (32 inserted, 0 duplicate text skipped)
Batch 5/24 done (32 inserted, 0 duplicate text skipped)
Batch 6/24 done (32 inserted, 0 duplicate text skipped)
Batch 7/24 done (32 inserted, 0 duplicate text skipped)
Batch 8/24 done (32 inserted, 0 duplicate text skipped)
Batch 9/24 done (32 inserted, 0 duplicate text skipped)
Batch 10/24 done (32 inserted, 0 duplicate text skipped)
Batch 11/24 done (32 inserted, 0 duplicate text skipped)
Batch 12/24 done (32 inserted, 0 duplicate text skipped)
Batch 13/24 done (32 inserted, 0 duplicate text skipped)
Batch 14/24 done (32 inserted, 0 duplicate text skipped)
Batch 15/24 done (32 inserted, 0 duplicate text skipped)
Batch 16/24 done (32 inserted, 0 duplicate text skipped)
Batch 17/24 done (32 inserted, 0 duplicate text skipped)
Batch 18/24 done (32 inserted, 0 duplica

In [55]:
document_path = Path("./documents/Инструкция_по_работе_с_Порталом_для_поставщика.pdf")
# 
page_count = pymupdf.open(document_path).page_count
json_string = pymupdf4llm.to_json(document_path, pages=[3, 4])
term_items = extract_table_glossary(json.loads(json_string), document_path)


In [56]:
term_items

[{'id': '1f64544b-2ef9-4d65-bed6-e6b6e7115332',
  'text': '223-ФЗ — Федеральный закон от 18 июля 2011 г. № 223-ФЗ «О закупках товаров, работ, \nуслуг отдельными видами юридических лиц»',
  'label': 'glossary',
  'path': 'documents/Инструкция_по_работе_с_Порталом_для_поставщика.pdf',
  'term': '223-ФЗ'},
 {'id': '79b19f50-1d1e-4a2d-86b0-f5871047c064',
  'text': '44-ФЗ — Федеральный закон от 5 апреля 2013 года № 44-ФЗ «О контрактной системе \nв сфере закупок товаров, работ, услуг для обеспечения государственных и \nмуниципальных нужд»',
  'label': 'glossary',
  'path': 'documents/Инструкция_по_работе_с_Порталом_для_поставщика.pdf',
  'term': '44-ФЗ'},
 {'id': '92709129-104c-48c3-9cdf-2c3dd35808b8',
  'text': '46-ФЗ — Федеральный закон от 08.03.2022 № 46-ФЗ «О внесении изменений в \nотдельные законодательные акты Российской Федерации»',
  'label': 'glossary',
  'path': 'documents/Инструкция_по_работе_с_Порталом_для_поставщика.pdf',
  'term': '46-ФЗ'},
 {'id': '58fcd771-9a71-4a69-bdca-cc14

In [57]:


embed_and_insert_batch(collection, term_items)

(42, 0)

In [58]:
doc = pymupdf4llm.to_markdown(document_path, pages=list(range(4, page_count)))


In [ ]:
splitted = splitter.split_text(doc)



In [63]:
splitted = splitted[4:]

In [64]:

items = [
                    {
                    "id": str(uuid4()),
                    "text": item,
                    "path": str(document_path),
                    "label": "manual",
                    "term": None,
                }
                for item in splitted
]
embed_and_insert_in_pool(items)

Batch 1/38 done (32 inserted, 0 duplicate text skipped)
Batch 2/38 done (32 inserted, 0 duplicate text skipped)
Batch 3/38 done (32 inserted, 0 duplicate text skipped)
Batch 4/38 done (32 inserted, 0 duplicate text skipped)
Batch 5/38 done (32 inserted, 0 duplicate text skipped)
Batch 6/38 done (32 inserted, 0 duplicate text skipped)
Batch 7/38 done (32 inserted, 0 duplicate text skipped)
Batch 8/38 done (32 inserted, 0 duplicate text skipped)
Batch 9/38 done (32 inserted, 0 duplicate text skipped)
Batch 10/38 done (32 inserted, 0 duplicate text skipped)
Batch 11/38 done (32 inserted, 0 duplicate text skipped)
Batch 12/38 done (32 inserted, 0 duplicate text skipped)
Batch 13/38 done (32 inserted, 0 duplicate text skipped)
Batch 14/38 done (32 inserted, 0 duplicate text skipped)
Batch 15/38 done (32 inserted, 0 duplicate text skipped)
Batch 16/38 done (32 inserted, 0 duplicate text skipped)
Batch 17/38 done (32 inserted, 0 duplicate text skipped)
Batch 18/38 done (32 inserted, 0 duplica

In [17]:
items = [
                    {
                    "id": str(uuid4()),
                    "text": item,
                    "path": str(document_path),
                    "label": "manual",
                    "term": None,
                }
                for item in splitted
]

In [18]:
embed_and_insert_in_pool(items)

Batch 1/24 done (32 inserted, 0 duplicate text skipped)
Batch 2/24 done (32 inserted, 0 duplicate text skipped)
Batch 3/24 done (32 inserted, 0 duplicate text skipped)
Batch 4/24 done (32 inserted, 0 duplicate text skipped)
Batch 5/24 done (32 inserted, 0 duplicate text skipped)
Batch 6/24 done (32 inserted, 0 duplicate text skipped)
Batch 7/24 done (32 inserted, 0 duplicate text skipped)
Batch 8/24 done (32 inserted, 0 duplicate text skipped)
Batch 9/24 done (32 inserted, 0 duplicate text skipped)
Batch 10/24 done (32 inserted, 0 duplicate text skipped)
Batch 11/24 done (32 inserted, 0 duplicate text skipped)
Batch 12/24 done (32 inserted, 0 duplicate text skipped)
Batch 13/24 done (32 inserted, 0 duplicate text skipped)
Batch 14/24 done (32 inserted, 0 duplicate text skipped)
Batch 15/24 done (32 inserted, 0 duplicate text skipped)
Batch 16/24 done (32 inserted, 0 duplicate text skipped)
Batch 17/24 done (32 inserted, 0 duplicate text skipped)
Batch 18/24 done (32 inserted, 0 duplica

In [19]:
document_path = Path("./documents/Инструкция_по_созданию_оферты_и_СТЕ.pdf")


page_count = pymupdf.open(document_path).page_count



In [23]:
json_string = pymupdf4llm.to_json(document_path, pages=[2])


In [26]:
term_items = extract_table_glossary(json.loads(json_string), document_path)

In [27]:
embed_and_insert_batch(collection, term_items)

(14, 0)

In [28]:
doc = pymupdf4llm.to_markdown(document_path, pages=list(range(3, page_count)))

In [34]:
splitted = splitter.split_text(doc)
items = [
                    {
                    "id": str(uuid4()),
                    "text": item,
                    "path": str(document_path),
                    "label": "manual",
                    "term": None,
                }
                for item in splitted
]

embed_and_insert_in_pool(items)

Batch 1/6 done (2 inserted, 0 duplicate text skipped)
Batch 2/6 done (32 inserted, 0 duplicate text skipped)
Batch 3/6 done (32 inserted, 0 duplicate text skipped)
Batch 4/6 done (32 inserted, 0 duplicate text skipped)
Batch 5/6 done (32 inserted, 0 duplicate text skipped)
Batch 6/6 done (32 inserted, 0 duplicate text skipped)


In [35]:
len(splitted)

162

In [36]:
document_path = Path("./documents/Инструкция_по_формированию_YML.pdf")


page_count = pymupdf.open(document_path).page_count
doc = pymupdf4llm.to_markdown(document_path, pages=list(range(3, page_count)))


In [37]:
splitted = splitter.split_text(doc)
items = [
                    {
                    "id": str(uuid4()),
                    "text": item,
                    "path": str(document_path),
                    "label": "manual",
                    "term": None,
                }
                for item in splitted
]

embed_and_insert_in_pool(items)

Batch 1/5 done (27 inserted, 0 duplicate text skipped)
Batch 2/5 done (32 inserted, 0 duplicate text skipped)
Batch 3/5 done (32 inserted, 0 duplicate text skipped)
Batch 4/5 done (32 inserted, 0 duplicate text skipped)
Batch 5/5 done (32 inserted, 0 duplicate text skipped)


In [38]:
document_path = Path("./documents/Инструкция_по_электронному_актированию.pdf")


page_count = pymupdf.open(document_path).page_count
json_string = pymupdf4llm.to_json(document_path, pages=[3])



In [41]:
term_items = extract_table_glossary(json.loads(json_string), document_path)
embed_and_insert_batch(collection, term_items)

(18, 0)

In [42]:
doc = pymupdf4llm.to_markdown(document_path, pages=list(range(4, page_count)))

In [43]:
splitted = splitter.split_text(doc)
items = [
                    {
                    "id": str(uuid4()),
                    "text": item,
                    "path": str(document_path),
                    "label": "manual",
                    "term": None,
                }
                for item in splitted
]

embed_and_insert_in_pool(items)

Batch 1/10 done (32 inserted, 0 duplicate text skipped)
Batch 2/10 done (32 inserted, 0 duplicate text skipped)
Batch 3/10 done (32 inserted, 0 duplicate text skipped)
Batch 4/10 done (32 inserted, 0 duplicate text skipped)
Batch 5/10 done (32 inserted, 0 duplicate text skipped)
Batch 6/10 done (32 inserted, 0 duplicate text skipped)
Batch 7/10 done (1 inserted, 0 duplicate text skipped)
Batch 8/10 done (32 inserted, 0 duplicate text skipped)
Batch 9/10 done (32 inserted, 0 duplicate text skipped)
Batch 10/10 done (32 inserted, 0 duplicate text skipped)
